# CoTraffic - training job on Google Colab

The training folder lives on Google Drive, so a disconnected session loses nothing:
run the notebook again with the same settings and training resumes from the latest
checkpoint.

In [ ]:
AGENT = "coop"          # ind, coop, coop_eco05, coop_eco, coop_full
NET = "grid4x4"         # grid4x4 or real_district
SEED = 1
TIMESTEPS = None
REPO_URL = "https://github.com/<team>/cotraffic.git"
REPO = "/content/cotraffic"
DRIVE_DIR = "/content/drive/MyDrive/cotraffic/training"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
link = f"{REPO}/results/training"
os.makedirs(f"{REPO}/results", exist_ok=True)
if not os.path.islink(link):
    os.symlink(DRIVE_DIR, link)

In [ ]:
import subprocess, sys
# SUMO and the RL stack; torch is already installed on the notebook image.
reqs = [l for l in open(f"{REPO}/requirements.txt") if l.strip() and not l.startswith(("#", "torch"))]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r.split("#")[0].strip() for r in reqs]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)

In [ ]:
os.environ["LIBSUMO_AS_TRACI"] = "1"
args = ["python", "-m", "cotraffic", "train", "--agent", AGENT, "--net", NET, "--seed", str(SEED)]
if TIMESTEPS:
    args += ["--timesteps", str(TIMESTEPS)]
!cd {REPO} && {" ".join(args)}